In [1]:

import os, urllib.request, zipfile
import numpy as np
import pandas as pd

if not os.path.exists("ml-100k"):
    urllib.request.urlretrieve(
        "https://files.grouplens.org/datasets/movielens/ml-100k.zip",
        "ml-100k.zip")
    zipfile.ZipFile("ml-100k.zip").extractall()

ratings = pd.read_csv(
    "ml-100k/u.data", sep="\t",
    names=["user", "item", "rating", "time"])

print(ratings.head())
print("Number of ratings:", len(ratings))

   user  item  rating       time
0   196   242       3  881250949
1   186   302       3  891717742
2    22   377       1  878887116
3   244    51       2  880606923
4   166   346       1  886397596
Number of ratings: 100000


This code downloads and extracts the MovieLens 100K dataset if it hasn't already been downloaded. It then uses Pandas to organize the data into a table containing the user ID, movie ID, rating, and timestamp. Finally, it displays the first five rows and the total number of ratings, which is 100,000.

In [2]:

def split_ratings(ratings, test_frac=0.10, seed=0):
    # Hide ratings without removing a user's or movie's last training rating.
    rng = np.random.default_rng(seed)
    is_test = np.zeros(len(ratings), dtype=bool)

    user_left = ratings.groupby("user").size().to_dict()
    item_left = ratings.groupby("item").size().to_dict()

    for pos in rng.permutation(len(ratings)):
        row = ratings.iloc[pos]
        user, item = row["user"], row["item"]

        if (rng.random() < test_frac
            and user_left[user] > 1
            and item_left[item] > 1):
            is_test[pos] = True
            user_left[user] -= 1
            item_left[item] -= 1

    return ratings.iloc[~is_test].copy(), ratings.iloc[is_test].copy()

train, test = split_ratings(ratings)

print("train:", len(train), "test:", len(test))
print("test fraction:", round(len(test) / len(ratings), 3))


train: 90032 test: 9968
test fraction: 0.1


This code randomly splits the movie ratings into training and testing sets, using about 90% for training and 10% for testing. It makes sure each user and movie still has at least one rating in the training set. It then displays how many ratings are in each set and the percentage used for testing.

In [3]:
N_USERS, N_ITEMS = 943, 1682

def to_matrix(df):
    """Return a user-by-movie matrix; missing ratings are np.nan."""
    R = np.full((N_USERS, N_ITEMS), np.nan)
    R[df["user"].to_numpy() - 1,
      df["item"].to_numpy() - 1] = df["rating"].to_numpy()
    return R

R = to_matrix(train)
known = ~np.isnan(R)
print("Matrix shape:", R.shape)
print("Known training ratings:", known.sum())

Matrix shape: (943, 1682)
Known training ratings: 90032


This code turns the training ratings into a matrix with users as rows and movies as columns. Ratings that are missing are marked as NaN instead of zero. It also counts how many training ratings are known.

In [4]:
truth = test["rating"].to_numpy()
u = test["user"].to_numpy() - 1
i = test["item"].to_numpy() - 1

This code saves the real hidden ratings and their user and movie positions. We will use them later to check how close our predictions are.

In [5]:
def rmse(pred, truth):
    """Root mean squared error, measured in stars."""
    return np.sqrt(np.mean((pred - truth)**2))

This function measures how far predicted ratings are from the actual ratings. It squares the errors, finds their average, and takes the square root. A smaller RMSE means better predictions.

In [6]:
mu = np.nanmean(R)
user_mean = np.nanmean(R, axis=1)
n_item = known.sum(axis=0)
item_mean = np.full(N_ITEMS, mu)
has_rating = n_item > 0
item_mean[has_rating] = (
    np.nansum(R, axis=0)[has_rating] / n_item[has_rating])
print("Overall average:", round(mu, 3))

Overall average: 3.53


This code calculates the overall average rating, each user’s average, and each movie’s average using only training data. If a movie has no ratings, it uses the overall average instead.

In [7]:
scores = {}
scores["always 3"] = rmse(np.full(len(test), 3.0), truth)
scores["overall average"] = rmse(np.full(len(test), mu), truth)
scores["user average"] = rmse(user_mean[u], truth)
scores["movie average"] = rmse(item_mean[i], truth)
for name, score in scores.items():
    print(f"{name:16s} {score:.4f}")

always 3         1.2516
overall average  1.1349
user average     1.0426
movie average    1.0284


This code compares four simple ways to predict ratings: always guessing 3 stars, using the overall average, using each user’s average, or using each movie’s average. It calculates RMSE for each method so we can compare them.

In [8]:
def bias_baseline(R):
    """Overall mean plus one user effect and one movie effect."""
    known = ~np.isnan(R)
    mu = np.nanmean(R)
    b_user = np.nanmean(R - mu, axis=1)
    residual = R - mu - b_user[:, None]
    n_item = known.sum(axis=0)
    b_item = np.zeros(R.shape[1])
    has_rating = n_item > 0
    b_item[has_rating] = (
        np.nansum(residual, axis=0)[has_rating] / n_item[has_rating])
    return mu, b_user, b_item

This function looks at whether users tend to rate higher or lower than average and whether movies tend to receive higher or lower ratings. It combines these effects to make a more personalized prediction.

In [9]:
mu, b_user, b_item = bias_baseline(R)
pred = np.clip(mu + b_user[u] + b_item[i], 1, 5)
scores["user + movie effects"] = rmse(pred, truth)
print(pd.Series(scores, name="test RMSE").round(4).sort_values())

user + movie effects    0.9545
movie average           1.0284
user average            1.0426
overall average         1.1349
always 3                1.2516
Name: test RMSE, dtype: float64


This code predicts the hidden ratings using both user and movie effects. It keeps predictions between 1 and 5 stars, then compares its RMSE with the other baselines. These results give us a benchmark for the SVD model in the next milestone.

Practice Exercises

Exercise 1

In [10]:
import numpy as np

tiny_R = np.array([
    [5, 3, np.nan],
    [4, np.nan, 1],
    [np.nan, 2, 2]
], dtype=float)

tiny_mu, tiny_user, tiny_item = bias_baseline(tiny_R)

print("Overall mean:", tiny_mu)
print("User biases:", tiny_user)
print("Movie biases:", tiny_item)


Overall mean: 2.8333333333333335
User biases: [ 1.16666667 -0.33333333 -0.83333333]
Movie biases: [ 1.25 -0.5  -0.75]



**Overall Mean:**

$$
\mu = \frac{5+3+4+1+2+2}{6} = 2.8333
$$

**User Biases:**

$$
b_{user} =
\begin{bmatrix}
1.1667\\
-0.3333\\
-0.8333
\end{bmatrix}
$$

**Movie Biases:**

$$
b_{item} =
\begin{bmatrix}
1.25 & -0.50 & -0.75
\end{bmatrix}
$$

I calculated the overall average by adding the known ratings and dividing by six, which gave me 2.8333. I then calculated each user's bias by subtracting the overall average from their individual average. Finally, I calculated the movie biases using the remaining differences. The Python results matched my hand calculations.

Exercise 2

In [11]:
# matrix all rating
R_all = to_matrix(ratings)

#user averages  all ratings
user_mean_all = np.nanmean(R_all, axis=1)

# hidden test
cheating_pred = user_mean_all[u]

# calculate the cheating RMSE
cheating_rmse = rmse(cheating_pred, truth)

print("Honest user-average RMSE:", round(scores["user average"], 4))
print("Cheating user-average RMSE:", round(cheating_rmse, 4))
print("Is cheating RMSE lower?",
      cheating_rmse < scores["user average"])

Honest user-average RMSE: 1.0426
Cheating user-average RMSE: 1.0322
Is cheating RMSE lower? True


Using all the ratings can make the RMSE lower because the model has access to ratings that were supposed to be hidden. This is misleading because the model is using test data to help make its predictions. It doesn't accurately show how well the model would predict new ratings.

Exercise 3

In [12]:
# creating a new split with seed=1
train1, test1 = split_ratings(ratings, seed=1)

# new training matrix
R1 = to_matrix(train1)
known1 = ~np.isnan(R1)

# new test ratings
truth1 = test1["rating"].to_numpy()
u1 = test1["user"].to_numpy() - 1
i1 = test1["item"].to_numpy() - 1

# finding averages
mu1 = np.nanmean(R1)
user_mean1 = np.nanmean(R1, axis=1)

n_item1 = known1.sum(axis=0)
item_mean1 = np.full(R1.shape[1], mu1)
has_rating1 = n_item1 > 0

item_mean1[has_rating1] = (
    np.nansum(R1, axis=0)[has_rating1] / n_item1[has_rating1]
)

# user and movie effects
mu_bias1, b_user1, b_item1 = bias_baseline(R1)

pred1 = np.clip(
    mu_bias1 + b_user1[u1] + b_item1[i1], 1, 5
)

# Calculate all five RMSE values
scores1 = {}

scores1["always 3"] = rmse(
    np.full(len(test1), 3.0), truth1
)
scores1["overall average"] = rmse(
    np.full(len(test1), mu1), truth1
)
scores1["user average"] = rmse(
    user_mean1[u1], truth1
)
scores1["movie average"] = rmse(
    item_mean1[i1], truth1
)
scores1["user + movie effects"] = rmse(pred1, truth1)

# Compare both seeds
comparison = pd.DataFrame({
    "Seed 0": pd.Series(scores),
    "Seed 1": pd.Series(scores1)
})

print(comparison.round(4))
print("\nRanking for seed 0:")
print(pd.Series(scores).sort_values().index.tolist())
print("\nRanking for seed 1:")
print(pd.Series(scores1).sort_values().index.tolist())


                      Seed 0  Seed 1
always 3              1.2516  1.2336
overall average       1.1349  1.1127
user average          1.0426  1.0345
movie average         1.0284  1.0159
user + movie effects  0.9545  0.9577

Ranking for seed 0:
['user + movie effects', 'movie average', 'user average', 'overall average', 'always 3']

Ranking for seed 1:
['user + movie effects', 'movie average', 'user average', 'overall average', 'always 3']


Changing the random seed changed the RMSE values slightly, but the ranking of the five models stayed the same. The user + movie effects model performed best for both seeds. This shows that results can vary depending on the training and testing split, so models should be compared using the same split for a fair evaluation.